# Adversarial face recognition — run on Colab

Runs the full pipeline with a GPU runtime. The VGGFace2 test images, the NN2 weights and the generated adversarial examples are expected on Google Drive (they are not part of the repository, see the README).

In [ ]:
!git clone https://github.com/RaffaeleRosanova01/adversarial-face-recognition.git
%cd adversarial-face-recognition
!pip install -q -r requirements.txt
!pip install -q --no-deps facenet-pytorch

## Data
Point `DRIVE_ROOT` to a Drive folder containing `data/` (test sets), `weights/` (NN2 and detector weights) and, optionally, `adversarial_data/` and `detector_dataset/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/adversarial-face-recognition'
for d in ['data', 'weights', 'adversarial_data', 'detector_dataset']:
    !ln -sfn "{DRIVE_ROOT}/{d}" {d}

To build the test sets from scratch from the VGGFace2 metadata and training archive:

In [ ]:
# !python scripts/prepare_data.py --meta data/identity_meta.csv --archive data/vggface2_train.tar.gz

## 1. Baseline

In [ ]:
!python scripts/baseline.py

## 2. Attacks on NN1 (white-box)

In [ ]:
!python scripts/run_attacks.py fgsm --save
!python scripts/run_attacks.py bim --max-iter 1 5 10 20 --save
!python scripts/run_attacks.py pgd --max-iter 1 5 10 20 40 --random-init 1 3 5 --save
!python scripts/run_attacks.py pgd --max-iter 1 5 10 20 40 --random-init 1 3 5 --targeted fixed --save
!python scripts/run_attacks.py cw --max-iter 1 5 10 20 --lr 0.01 0.1 --save
!python scripts/run_attacks.py deepfool --max-iter 1 3 5 10 20 50 100 200 --save

## 3. Transferability to NN2

In [ ]:
!python scripts/transferability.py FGSM
!python scripts/transferability.py PGD_Targeted_Fixed_nri1_iter40 --target 2026
!python scripts/transferability.py DeepFool --flat

## 4. Detector training and defences

In [ ]:
!python scripts/train_detector.py --backbone efficientnet
!python scripts/evaluate_defenses.py --backbone efficientnet

## 5. Adaptive attack against detector + NN1

In [ ]:
!python scripts/adaptive_attack.py --steps 20 --eps 0.005 0.01 0.03 0.1 --lams 0 1